In [ ]:
import torch
import torch.nn as nn
import numpy as np


In [ ]:
"""
실습 목표
1) 완전한 Tensor Fusion: Z = ⊗_m z_m,  h = W · Z + b
2) Low-rank weight decomposition(CP/LMF 스타일):
   W ≈ Σ_{i=1..r} ⊗_m w_m^(i)   (출력 차원 d_h를 공유하는 형태)
   - (A) W를 복원(reconstruct)해서 h 계산
   - (B) (옵션) 논문 Eq.(6)/(7)처럼 텐서 Z를 만들지 않고 효율적으로 h 계산

주의:
- 여기서는 "append 1"을 포함해서 TFN(부분집합 상호작용) 철학도 반영합니다.
- 모든 코드는 CPU에서 바로 실행 가능.
"""

In [35]:
torch.manual_seed(0)

# -----------------------------
# 0) 설정: bimodal 예시 (M=2)
# -----------------------------
B = 4          # batch size
d_g = 3        # graph vector dim
d_d = 5        # desc vector dim
d_h = 7        # output dim
r = 4          # low-rank

# unimodal vectors (batch)
g = torch.randn(B, d_g)
d = torch.randn(B, d_d)
print(g.shape)
print(g)

print(d.shape)
print(d)


torch.Size([4, 3])
tensor([[ 1.5410, -0.2934, -2.1788],
        [ 0.5684, -1.0845, -1.3986],
        [ 0.4033,  0.8380, -0.7193],
        [-0.4033, -0.5966,  0.1820]])
torch.Size([4, 5])
tensor([[ 0.4681, -0.1577,  1.4437,  0.2660, -0.1740],
        [-0.6787,  0.9383,  0.4889, -0.6731,  0.8728],
        [ 1.0554,  0.1778, -0.5181, -0.3067, -1.5810],
        [ 1.7066, -0.4462,  0.7440,  1.5210,  3.4105]])


In [36]:
# TFN extension: append 1
ones = torch.ones(B, 1)
g_hat = torch.cat([g, ones], dim=1)   # (B, d_g+1)
d_hat = torch.cat([d, ones], dim=1)   # (B, d_d+1)
print(g_hat.shape)
print(g_hat)

print(d_hat.shape)
print(d_hat)

torch.Size([4, 4])
tensor([[ 1.5410, -0.2934, -2.1788,  1.0000],
        [ 0.5684, -1.0845, -1.3986,  1.0000],
        [ 0.4033,  0.8380, -0.7193,  1.0000],
        [-0.4033, -0.5966,  0.1820,  1.0000]])
torch.Size([4, 6])
tensor([[ 0.4681, -0.1577,  1.4437,  0.2660, -0.1740,  1.0000],
        [-0.6787,  0.9383,  0.4889, -0.6731,  0.8728,  1.0000],
        [ 1.0554,  0.1778, -0.5181, -0.3067, -1.5810,  1.0000],
        [ 1.7066, -0.4462,  0.7440,  1.5210,  3.4105,  1.0000]])


In [37]:
# einsum 테스트
print(g_hat[0,:])
print(d_hat[0,:])
print(torch.matmul(g_hat[0,:].view(-1,1), d_hat[0,:].view(1,-1)))

tensor([ 1.5410, -0.2934, -2.1788,  1.0000])
tensor([ 0.4681, -0.1577,  1.4437,  0.2660, -0.1740,  1.0000])
tensor([[ 0.7213, -0.2430,  2.2247,  0.4100, -0.2681,  1.5410],
        [-0.1374,  0.0463, -0.4236, -0.0781,  0.0510, -0.2934],
        [-1.0199,  0.3436, -3.1454, -0.5797,  0.3790, -2.1788],
        [ 0.4681, -0.1577,  1.4437,  0.2660, -0.1740,  1.0000]])


In [ ]:
# -------------------------------------------------------
# 1) 완전한 Tensor Fusion (explicit Z and W)
# 각 배치에 대해 outer product를 쌓은 것
# -------------------------------------------------------
# Z: outer product => (B, dg1, dd1)
Z = torch.einsum("bi,bj->bij", g_hat, d_hat)
print(Z.shape)
print(Z)

torch.Size([4, 4, 6])
tensor([[[ 0.7213, -0.2430,  2.2247,  0.4100, -0.2681,  1.5410],
         [-0.1374,  0.0463, -0.4236, -0.0781,  0.0510, -0.2934],
         [-1.0199,  0.3436, -3.1454, -0.5797,  0.3790, -2.1788],
         [ 0.4681, -0.1577,  1.4437,  0.2660, -0.1740,  1.0000]],

        [[-0.3858,  0.5333,  0.2779, -0.3826,  0.4961,  0.5684],
         [ 0.7361, -1.0176, -0.5302,  0.7300, -0.9466, -1.0845],
         [ 0.9493, -1.3122, -0.6837,  0.9414, -1.2207, -1.3986],
         [-0.6787,  0.9383,  0.4889, -0.6731,  0.8728,  1.0000]],

        [[ 0.4257,  0.0717, -0.2090, -0.1237, -0.6377,  0.4033],
         [ 0.8844,  0.1490, -0.4342, -0.2570, -1.3249,  0.8380],
         [-0.7591, -0.1279,  0.3726,  0.2206,  1.1371, -0.7193],
         [ 1.0554,  0.1778, -0.5181, -0.3067, -1.5810,  1.0000]],

        [[-0.6884,  0.1800, -0.3001, -0.6135, -1.3756, -0.4033],
         [-1.0182,  0.2662, -0.4439, -0.9075, -2.0348, -0.5966],
         [ 0.3107, -0.0812,  0.1354,  0.2769,  0.6208,  0.1820

In [39]:
# Full Weight + Bias
dg1 = d_g + 1
dd1 = d_d + 1

# W_full: (dg1, dd1, d_h), b: (d_h,)
W_full = torch.randn(dg1, dd1, d_h)     # weight
b = torch.randn(d_h)                    # bias

print(W_full.shape)
print(b.shape)

# W*Z + b
# h_full[b,k] = Σ_i Σ_j W[i,j,k] * Z[b,i,j] + b[k]
h_full = torch.einsum("bij,ijk->bk", Z, W_full) + b
print()
print("h_full shape:", h_full.shape)

torch.Size([4, 6, 7])
torch.Size([7])

h_full shape: torch.Size([4, 7])


In [40]:
# -------------------------------------------------------
# 2) Low-rank weight decomposition (LMF-style)
#    W_lowrank = Σ_{i=1..r} (w_g[i] ⊗ w_d[i])   (output dim 공유)
#    where w_g[i] ∈ R^{dg1 × d_h},  w_d[i] ∈ R^{dd1 × d_h}
# -------------------------------------------------------
w_g = torch.randn(r, dg1, d_h)  # modality-specific factors for g
w_d = torch.randn(r, dd1, d_h)  # modality-specific factors for d

print(w_g.shape)
print(w_d.shape)

torch.Size([4, 4, 7])
torch.Size([4, 6, 7])


In [ ]:
# (A) W를 실제로 복원해서 계산 (비효율적이지만 "decomposition" 확인용)
# W_rec: (dg1, dd1, d_h)
W_rec = torch.zeros(dg1, dd1, d_h)
for i in range(r):
    # outer product over non-shared dims only:
    # (dg1, d_h) and (dd1, d_h) -> (dg1, dd1, d_h)
    W_rec += torch.einsum("ah,bh->abh", w_g[i], w_d[i])

h_rec = torch.einsum("bij,ijk->bk", Z, W_rec) + b

print("W_rec shape:", W_rec.shape)
print("h_rec shape:", h_rec.shape)

W_rec shape: torch.Size([4, 6, 7])
h_rec shape: torch.Size([4, 7])


In [47]:
# (B) 효율적 계산: 텐서 Z와 W를 만들지 않고 Eq.(7) 방식으로
# bimodal Eq.(7) 해석:
#   h = (Σ_i (w_g[i]^T z_g))  ∘  (Σ_i (w_d[i]^T z_d))   ?  
# (주의: 논문은 rank-sum과 hadamard 순서가 바뀐 형태도 제시)
#
# 논문 Eq.(8) 스타일(실무 구현과 유사):
#   먼저 각 모달별로 rank별 projection을 만들고,
#   rank별로 모달 간 hadamard 한 뒤,
#   rank 축을 sum:
#
#   fusion_g[b,i,h] = Σ_a g_hat[b,a] * w_g[i,a,h]
#   fusion_d[b,i,h] = Σ_b d_hat[b,b] * w_d[i,b,h]
#   h_eff[b,h] = Σ_i fusion_g[b,i,h] * fusion_d[b,i,h] + b[h]
fusion_g = torch.einsum("bd,rdh->brh", g_hat, w_g)  # (B, r, d_h)
fusion_d = torch.einsum("bd,rdh->brh", d_hat, w_d)  # (B, r, d_h)
h_eff = (fusion_g * fusion_d).sum(dim=1) + b        # (B, d_h)
print("h_eff shape:", h_eff.shape)


h_eff shape: torch.Size([4, 7])


In [48]:



# -------------------------------------------------------
# 3) 검증: (A) 복원 방식과 (B) 효율 방식이 같은지
# -------------------------------------------------------
max_diff = (h_rec - h_eff).abs().max().item()
print("max |h_rec - h_eff| =", max_diff)

# -------------------------------------------------------
# 4) (옵션) "W_full"을 low-rank로 근사해보고 싶다면?
#    실제로는 W_full을 학습하지 않고 w_g, w_d를 학습하지만,
#    여기서는 감각용으로 W_full과 W_rec의 차이를 봅니다.
# -------------------------------------------------------
approx_error = (W_full - W_rec).pow(2).mean().sqrt().item()
print("RMSE(W_full, W_rec) =", approx_error)

"""
출력 해석
- h_full: 완전한 텐서퓨전(임의 W_full) 결과
- h_rec : low-rank factor로 복원한 W_rec로 계산한 결과
- h_eff : 논문 Eq.(8) 스타일의 효율 계산 결과 (h_rec과 같아야 함)
- max_diff가 1e-6 수준이면 실습이 올바른 것.
"""

max |h_rec - h_eff| = 2.384185791015625e-06
RMSE(W_full, W_rec) = 2.275400161743164


'\n출력 해석\n- h_full: 완전한 텐서퓨전(임의 W_full) 결과\n- h_rec : low-rank factor로 복원한 W_rec로 계산한 결과\n- h_eff : 논문 Eq.(8) 스타일의 효율 계산 결과 (h_rec과 같아야 함)\n- max_diff가 1e-6 수준이면 실습이 올바른 것.\n'

In [ ]:
# 검증까지 한 번에 돌리는 최소 예시
import torch
torch.manual_seed(0)

B, d_g, d_d, d_h, r = 4, 3, 5, 7, 4
g = torch.randn(B, d_g)
d = torch.randn(B, d_d)

ones = torch.ones(B, 1)
g_hat = torch.cat([g, ones], dim=1)   # (B, dg1)
d_hat = torch.cat([d, ones], dim=1)   # (B, dd1)

dg1, dd1 = d_g+1, d_d+1

# explicit Z
Z = torch.einsum("bi,bj->bij", g_hat, d_hat)  # (B, dg1, dd1)

# low-rank factors
w_g = torch.randn(r, dg1, d_h)
w_d = torch.randn(r, dd1, d_h)
b = torch.randn(d_h)

# (A) reconstruct W then compute
W_rec = torch.zeros(dg1, dd1, d_h)
for i in range(r):
    W_rec += torch.einsum("ah,bh->abh", w_g[i], w_d[i])
h_rec = torch.einsum("bij,ijk->bk", Z, W_rec) + b

# (B) efficient compute (fixed!)
fusion_g = torch.einsum("bd,rdh->brh", g_hat, w_g)
fusion_d = torch.einsum("bd,rdh->brh", d_hat, w_d)
h_eff = (fusion_g * fusion_d).sum(dim=1) + b

print("max |h_rec - h_eff| =", (h_rec - h_eff).abs().max().item())

max |h_rec - h_eff| = 3.814697265625e-06


In [2]:
li = ['smiles', 'use_MORSE_129', 'use_MORSE_161', 'use_GETAWAY_105', 'use_GETAWAY_133', 'use_GETAWAY_025', 'use_RadiusOfGyration', 'use_MORSE_007', 'use_GETAWAY_085', 'use_GETAWAY_263', 'use_GETAWAY_086', 'use_MORSE_194', 'use_GETAWAY_045', 'use_MORSE_065', 'use_MORSE_193', 'use_MORSE_094', 'use_USRCAT_057', 'use_USRCAT_000', 'use_GETAWAY_005', 'use_GETAWAY_093', 'use_GETAWAY_007', 'use_USRCAT_007', 'use_GETAWAY_002', 'use_GETAWAY_257', 'use_USR_006', 'use_USRCAT_054', 'use_GETAWAY_113', 'use_MORSE_068', 'use_MORSE_097', 'use_MORSE_033', 'use_GETAWAY_067', 'use_GETAWAY_107', 'use_GETAWAY_047', 'use_USRCAT_058', 'use_MORSE_222', 'use_USRCAT_055', 'use_MORSE_167', 'use_USRCAT_043', 'use_GETAWAY_046', 'use_GETAWAY_000', 'use_USRCAT_052', 'use_MORSE_135', 'use_GETAWAY_066', 'use_USRCAT_051', 'use_GETAWAY_084']
len(li)

45